In [4]:
import json
from datetime import datetime,date
from decimal import Decimal
def to_serializable(o):
    if isinstance(o,(date,datetime)):
        return o.isoformat()
    if isinstance(o,Decimal):
        return float(round(o,2))
    if isinstance(o,set):
        return sorted(o)
    raise TypeError(f"不可序列化:{type(o)}")
obj = {
    'name':'xiaoming',
    'score':92,
    'tags':['qinfen',None,None],
    'born':date(2005,3,14),
    'balance':Decimal('19.90'),
}
s = json.dumps(obj,ensure_ascii=False,indent=2,sort_keys=True,default=to_serializable)
print(s)
print(json.dumps(obj,ensure_ascii=False,default=to_serializable))
with open('data.json','w',encoding='utf-8') as f:
    json.dump(obj,f,ensure_ascii=False,default=to_serializable)
with open('data.json',encoding='utf-8') as f:
    data = json.load(f)
print(data['name'],type(data["score"]))
with open('events.jsonl','w',encoding='utf-8') as f:
    for i in range(3):
        f.write(json.dumps({"id":i,"ts":datetime.now().isoformat()},ensure_ascii=False)+"\n")
errors = []
with open("events.jsonl",encoding="utf-8") as f:
    for lineno,line in enumerate(f,1):
        line = line.strip()
        if not line:
            continue
        try:
            rec = json.loads(line)
        except json.JSONDecodeError as e:
            errors.append((lineno,str(e)))
            continue
        print(rec)

{
  "balance": 19.9,
  "born": "2005-03-14",
  "name": "xiaoming",
  "score": 92,
  "tags": [
    "qinfen",
    null,
    null
  ]
}
{"name": "xiaoming", "score": 92, "tags": ["qinfen", null, null], "born": "2005-03-14", "balance": 19.9}
xiaoming <class 'int'>
{'id': 0, 'ts': '2026-09-30T11:12:18.935496'}
{'id': 1, 'ts': '2026-09-30T11:12:18.935496'}
{'id': 2, 'ts': '2026-09-30T11:12:18.935496'}


In [7]:
import csv
from pathlib import Path
ROWS = [
    {'姓名':'小明','成绩':92,'备注':'优秀,commendable'},
    {'姓名':'小红','成绩':88,'备注':None},
]
with open("scores.csv","w",encoding="utf-8-sig",newline="") as f:
    w = csv.DictWriter(f,fieldnames=["姓名","成绩","备注"],quoting=csv.QUOTE_MINIMAL)
    w.writeheader()
    w.writerows(ROWS)
with open("scores.csv",encoding="utf-8-sig",newline="") as f:
    for row in csv.DictReader(f):
        print(row["姓名"],int(row["成绩"]),row["备注"])
with open("data.tsv","w",encoding="utf-8",newline="") as f:
    csv.writer(f,delimiter="\t").writerows([[1,2],[3,4]])
with open("q.csv","w",encoding="utf-8",newline="") as f:
    csv.writer(f,quoting=csv.QUOTE_ALL).writerow(["a,b","c"])
def csv_to_json(src,dst,batch=1000):
    n = 0
    with open(src,encoding="utf-8-sig",newline="") as fin, \
         open(dst,"w",encoding="utf-8") as fout:
         for row in csv.DictReader(fin):
             fout.write(json.dumps(row,ensure_ascii=False)+"\n")
             n += 1
             if n % batch == 0:
                 fout.flush()
    return n


小明 92 优秀,commendable
小红 88 


In [9]:
import pickle,shelve
from pathlib import Path
obj = {"模型":[1,2,3],"日期":date.today()}
Path("cache.pki").write_bytes(pickle.dumps(obj))
obj2 = pickle.loads(Path("cache.pkl").read_bytes())
print(obj2)
with open("cache.pkl","wb") as f:
    pickle.dump(obj,f,protocol=pickle.HIGHEST_PROTOCOL)
with shelve.open("db") as db:
    db["user:1"] = {"name":"小明","score":92}
    print(db.keys())
with shelve.open("db",writeback=True) as db:
    u = db["user:1"]
    u["score"] = 100
    db["user:1"] = u

{'模型': [1, 2, 3], '日期': datetime.date(2026, 9, 30)}
KeysView(<shelve.DbfilenameShelf object at 0x0000027E3DDEE090>)


In [7]:
import os
from pathlib import Path
import tempfile
def save_atomic(path,content,encoding="utf-8"):
    path = Path(path)
    path.parent.mkdir(parents=True,exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    try:
        with open(tmp,"w",encoding=encoding) as f:
            f.write(content)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp,path)
    finally:
        tmp.unlink(missing_ok=True)
save_atomic("config.json",'{"port":8080}\n')
with tempfile.NamedTemporaryFile("w+",encoding="utf-8",delete=False) as tf:
    tf.write("临时数据")
    tmp_path = tf.name
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "work.txt"
    p.write_text("hello",encoding="utf-8")
    print(p.read_text(encoding="utf-8"))

hello


In [ ]:
import numpy as np
def random_walk_loop(N,seed=42):
    rng = np.random.default_rng(seed)
    steps = rng.choice([-1,1],size=N)
    pos = np.zeros(N+1,dtype=np.int64)
    for i in range(N):
        